# Convert ipynb to HTML/PDF in Colab

#📘 Notebook Converter

In [1]:
import os
import re
import shutil
import subprocess
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files

# =========================================================
# Config
# =========================================================
WORKSPACE = Path("/content/conversion_workspace")

# =========================================================
# State
# =========================================================
state = {
    "original_uploaded_name": None,
    "workspace_file": None,
    "workspace_base": None
}

# =========================================================
# Helpers
# =========================================================
def run_cmd(cmd, cwd=None):
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=cwd)
    return result.returncode == 0, result.stdout, result.stderr

def pip_install(pkg):
    return run_cmd(["pip", "install", "-q", pkg])

def sanitize_filename(filename):
    base, ext = os.path.splitext(filename)
    safe_base = re.sub(r"[^A-Za-z0-9._-]+", "_", base).strip("._")
    if not safe_base:
        safe_base = "uploaded_notebook"
    return safe_base + ext.lower()

def ensure_file_exists(path):
    return Path(path).exists() and Path(path).is_file()

def reset_workspace():
    if WORKSPACE.exists():
        shutil.rmtree(WORKSPACE)
    WORKSPACE.mkdir(parents=True, exist_ok=True)

# =========================================================
# UI
# =========================================================
title = widgets.HTML("""
<div style="border:1px solid #ddd; border-radius:10px; padding:14px; background:#f8f9fa;">
  <h2 style="margin:0 0 8px 0;">📘 Convert Jupyter Notebook</h2>
  <div style="margin:0;">
    <b>Step 1:</b> Upload your <code>.ipynb</code> file.<br>
    <b>Step 2:</b> Choose your desired output format from the dropdown.<br>
    <b>Step 3:</b> Click Convert.<br><br>
    <i>Note: Each upload automatically clears the temporary workspace to prevent conflicts.</i>
  </div>
</div>
""")

# Streamlined dropdown options
conversion_dropdown = widgets.Dropdown(
    options=[
        ("1. HTML", "html_default"),
        ("2.1. JupyterLab style (PDF)", "pdf_lab"),
        ("2.2. Classic notebook style (PDF)", "pdf_classic")
    ],
    value="pdf_lab",
    description="Select Format:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px")
)

uploader = widgets.FileUpload(
    accept=".ipynb",
    multiple=False,
    description=" Upload File",
    icon="upload",
    button_style="info",
    layout=widgets.Layout(width="220px", height="42px")
)

convert_btn = widgets.Button(
    description=" Convert Now",
    icon="cog",
    button_style="success",
    layout=widgets.Layout(width="180px", height="42px")
)

file_label = widgets.HTML("<b>No file uploaded yet.</b>")
log_out = widgets.Output()

display(widgets.VBox([
    title,
    conversion_dropdown,
    widgets.HBox([uploader, convert_btn]),
    file_label,
    log_out
]))

# =========================================================
# Upload handling
# =========================================================
def handle_upload(change):
    if not uploader.value:
        return

    with log_out:
        clear_output()
        print("Preparing isolated workspace...")

    reset_workspace()

    state["original_uploaded_name"] = None
    state["workspace_file"] = None
    state["workspace_base"] = None

    uploaded_items = uploader.value
    if isinstance(uploaded_items, dict):
        first_key = next(iter(uploaded_items))
        uploaded_file = uploaded_items[first_key]
        original_name = uploaded_file["metadata"]["name"] if "metadata" in uploaded_file else first_key
        content = uploaded_file["content"]
    else:
        uploaded_file = uploaded_items[0]
        original_name = uploaded_file["name"]
        content = uploaded_file["content"]

    if not original_name.lower().endswith(".ipynb"):
        with log_out:
            clear_output()
            print("Error: Please upload a .ipynb file only.")
        file_label.value = "<b style='color:red;'>Invalid file. Please upload a .ipynb notebook.</b>"
        return

    safe_name = sanitize_filename(original_name)
    workspace_file = WORKSPACE / safe_name

    with open(workspace_file, "wb") as f:
        f.write(content)

    state["original_uploaded_name"] = original_name
    state["workspace_file"] = str(workspace_file)
    state["workspace_base"] = str(workspace_file.with_suffix(""))

    file_label.value = (
        f"<div style='padding:10px; background:#e2f0d9; border-radius:5px; margin-top:10px;'>"
        f"✅ <b>Successfully Uploaded:</b> {original_name}<br>"
        f"📁 <b>Temporary Workspace:</b> {WORKSPACE}"
        f"</div>"
    )

    with log_out:
        clear_output()
        print("Upload successful.")
        print(f"Workspace fully cleared and ready for: {original_name}")

uploader.observe(handle_upload, names="value")

# =========================================================
# Convert handling
# =========================================================
def convert_action(_):
    with log_out:
        clear_output()

        if not state["workspace_file"]:
            print("Please upload a .ipynb file first!")
            return

        ipynb_file = Path(state["workspace_file"])
        base_name = Path(state["workspace_base"])
        selection = conversion_dropdown.value

        # Cleaned up routing logic
        if selection == "html_default":
            output_format = "html"
            template_name = "lab"
        elif selection == "pdf_lab":
            output_format = "webpdf"
            template_name = "lab"
        elif selection == "pdf_classic":
            output_format = "webpdf"
            template_name = "classic"

        print(f"Workspace notebook: {ipynb_file}")
        print(f"Processing target: {output_format.upper()} (Template: {template_name})")

        print("\nInstalling nbconvert and WebPDF dependencies...")
        ok, out, err = pip_install("nbconvert[webpdf]")
        if not ok:
            print("Failed to install nbconvert.")
            print(err if err else out)
            return

        if output_format == "webpdf":
            print("Installing browser dependencies for PDF rendering...")
            run_cmd(["playwright", "install-deps"])
            run_cmd(["playwright", "install", "chromium"])

        print(f"\nConverting notebook to {output_format.upper()}...")
        cmd = [
            "jupyter", "nbconvert",
            "--to", output_format,
            "--template", template_name,
            ipynb_file.name
        ]

        if output_format == "html":
            cmd.append("--embed-images")

        ok, out, err = run_cmd(cmd, cwd=str(WORKSPACE))

        if not ok:
            print(f"Failed to convert notebook to {output_format.upper()}.")
            print(err if err else out)
            return

        if output_format == "webpdf":
            output_file = WORKSPACE / f"{base_name.name}.pdf"
        else:
            output_file = WORKSPACE / f"{base_name.name}.html"

        if not ensure_file_exists(output_file):
            print(f"Expected output file not found: {output_file.name}")
            return

        print(f"✅ Conversion successful! Downloading {output_file.name}...")
        files.download(str(output_file))

convert_btn.on_click(convert_action)